In [ ]:
import matplotlib.pyplot as plt


VARIANT_NAME = "ResNet2D_Baseline"
EXPERIMENTS_ROOT = os.path.join(root_path, "experiments", VARIANT_NAME)

# CONFIGURATION FOLDER
RUN_ID = get_next_run_id(EXPERIMENTS_ROOT)
checkpoint_dir = os.path.join(EXPERIMENTS_ROOT, RUN_ID)
best_model_path = os.path.join(checkpoint_dir, "best_model.pth")
plot_path = os.path.join(checkpoint_dir, "training_curves.png") # Path per l'immagine

os.makedirs(checkpoint_dir, exist_ok=True)

print(f"==================================================")
print(f" EXPERIMENT STARTED: {VARIANT_NAME}")
print(f" AUTO-ASSIGNED ID:   {RUN_ID}")
print(f" OUTPUT FOLDER:      {checkpoint_dir}")
print(f"==================================================")

# HYPERPARAMETERS
PATIENCE = 7
EARLY_STOPPING_DELTA = 0.001
LEARNING_RATE = 1e-4
NUM_EPOCHS = 50

# Initialize model
print("Initializing Model...")
model = ResNet2D_Baseline(num_classes=NUM_CLASSES, pretrained=True).to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

train_loss_hist = []
val_loss_hist = []
train_acc_hist = []
val_acc_hist = []

best_val_acc = 0.0
min_val_loss = float('inf')
early_stopping_counter = 0

print(f"\nStarting Training loop for {NUM_EPOCHS} epochs...")
print(f"Stop condition: No validation loss improvement (delta={EARLY_STOPPING_DELTA}) for {PATIENCE} epochs.\n")

for epoch in range(NUM_EPOCHS):
    print(f"Epoch {epoch+1}/{NUM_EPOCHS}")

    # TRAIN STEP
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE)

    # VALIDATION STEP
    val_loss, val_acc = validate(model, val_loader, criterion, DEVICE)

    train_loss_hist.append(train_loss)
    val_loss_hist.append(val_loss)
    train_acc_hist.append(train_acc)
    val_acc_hist.append(val_acc)

    # Print metrics
    print(f" -> Train Loss: {train_loss:.4f} | Acc: {train_acc:.2f}%")
    print(f" -> Val Loss:   {val_loss:.4f} | Acc: {val_acc:.2f}%")

    # CHECKPOINT SAVING
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), best_model_path)
        print(f"    >>> New Best Validation Accuracy! Model saved.")

    # EARLY STOPPING LOGIC
    if val_loss < (min_val_loss - EARLY_STOPPING_DELTA):
        min_val_loss = val_loss
        early_stopping_counter = 0
    else:
        early_stopping_counter += 1
        print(f"!!! No improvement. Patience: {early_stopping_counter}/{PATIENCE}")

        if early_stopping_counter >= PATIENCE:
            print(f"\nEARLY STOPPING TRIGGERED.")
            print("   Stopping training to prevent overfitting.")
            break

print("\n" + "="*50)
print(" TRAINING COMPLETE")
print("="*50)

print("Generating training plots...")

plt.figure(figsize=(12, 5))

# Plot Loss
plt.subplot(1, 2, 1)
plt.plot(train_loss_hist, label='Train Loss', color='blue')
plt.plot(val_loss_hist, label='Val Loss', color='orange')
plt.title(f'{VARIANT_NAME} - Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

# Plot Accuracy
plt.subplot(1, 2, 2)
plt.plot(train_acc_hist, label='Train Acc', color='green')
plt.plot(val_acc_hist, label='Val Acc', color='red')
plt.title(f'{VARIANT_NAME} - Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy (%)')
plt.legend()
plt.grid(True)

plt.show()